# BERT + LDA Ensemble for Suicide Risk Detection

This notebook builds a **two-branch ensemble** that combines:

| Branch | Model | Feature type |
|--------|-------|--------------|
| A | Multilingual BERT (`bert-base-multilingual-cased`) | Contextual [CLS] embeddings |
| B | LDA (scikit-learn) | Topic-probability vectors |

Both branches feed into a **logistic regression meta-classifier** (late fusion).

**Task:** binary classification — `suicide_risk` (1) vs `no_risk` (0).

> **Note:** The corpus below is synthetic and illustrative. Replace it with your real annotated dataset before production use.

## 0. Dependencies

In [ ]:
# Uncomment to install in a fresh environment
# !pip install torch transformers scikit-learn numpy pandas matplotlib

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay
)
from sklearn.pipeline import Pipeline

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"cuda" if torch.cuda.is_available() else "cpu"}')

## 1. Synthetic Corpus

Replace `texts` / `labels` with your real dataset.  
Labels: **1** = suicide risk indicator, **0** = no risk.

In [ ]:
texts = [
    # --- Risk (label=1) ---
    "No aguanto más, creo que sería mejor desaparecer para siempre.",
    "Estoy pensando en hacerme daño, ya no veo otra salida.",
    "La vida no tiene sentido, no quiero seguir viviendo.",
    "Me siento completamente solo y sin esperanza de mejorar.",
    "He estado pensando en suicidarme, ya lo tengo planeado.",
    "Todo sería más fácil si simplemente dejara de existir.",
    "No puedo soportar este dolor, quiero que todo termine.",
    "Me despedí de mis amigos esta semana, no creo que los vuelva a ver.",
    "Ya no me importa nada, he regalado todas mis cosas.",
    "Cada noche me pregunto si mañana seguiré aquí.",
    "Siento que soy una carga para todos los que me rodean.",
    "He buscado formas de quitarme la vida, no puedo más.",
    "El dolor es insoportable y no veo ningún futuro.",
    "Nadie me echará de menos si desaparezco.",
    "Me siento atrapado sin salida y quiero acabar con todo.",
    # --- No Risk (label=0) ---
    "Hoy tuve un día difícil en el trabajo pero mañana será mejor.",
    "Me siento triste a veces, pero tengo apoyo de mi familia.",
    "El estrés laboral es alto pero estoy aprendiendo a manejarlo.",
    "Tuve una discusión con mi pareja, necesito hablar con alguien.",
    "Me preocupa mi salud pero los médicos dicen que estoy bien.",
    "A veces me siento solo, pero disfruto de mis aficiones.",
    "Tuve una semana dura pero el fin de semana descansé.",
    "Siento ansiedad ante los exámenes pero confío en prepararme.",
    "Mi estado de ánimo varía, hablo con mi psicólogo cada semana.",
    "Estoy pasando por un divorcio difícil pero tengo apoyo.",
    "Me cuesta dormir últimamente debido al estrés.",
    "Tuve pensamientos negativos pero los superé hablando con amigos.",
    "Estoy triste por la pérdida de mi mascota pero sigo adelante.",
    "Me siento agotado pero sé que las cosas mejorarán.",
    "A veces lloro sin motivo aparente pero me recupero pronto.",
]

labels = [1]*15 + [0]*15

df = pd.DataFrame({'text': texts, 'label': labels})
print(df['label'].value_counts().rename({1: 'risk', 0: 'no_risk'}))
df.head(4)

## 2. Preprocessing

In [ ]:
def clean(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean'] = df['text'].apply(clean)

# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'].tolist(), df['label'].tolist(),
    test_size=0.25, random_state=42, stratify=df['label']
)
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')

## 3. Branch A — BERT [CLS] Embeddings

We use `bert-base-multilingual-cased` (supports Spanish out of the box).  
For production consider `dccuchile/bert-base-spanish-wwm-cased` (BETO).

In [ ]:
BERT_MODEL = 'bert-base-multilingual-cased'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Loading {BERT_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()
print('Done.')

In [ ]:
@torch.no_grad()
def get_bert_embeddings(texts: list[str], batch_size: int = 8) -> np.ndarray:
    """Return mean-pooled last-hidden-state embeddings (shape: n x 768)."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(device)
        output = bert_model(**encoded)
        # Mean pool over tokens (excluding padding)
        attention_mask = encoded['attention_mask'].unsqueeze(-1).float()
        token_embeddings = output.last_hidden_state
        summed = (token_embeddings * attention_mask).sum(dim=1)
        counts = attention_mask.sum(dim=1)
        mean_pooled = (summed / counts).cpu().numpy()
        all_embeddings.append(mean_pooled)
    return np.vstack(all_embeddings)

print('Encoding training texts with BERT ...')
X_bert_train = get_bert_embeddings(X_train)
print('Encoding test texts with BERT ...')
X_bert_test  = get_bert_embeddings(X_test)

print(f'BERT train shape: {X_bert_train.shape}')  # (n_train, 768)
print(f'BERT test shape : {X_bert_test.shape}')

## 4. Branch B — LDA Topic Features

We use `n_topics=5` to capture latent themes (hopelessness, isolation, planning, everyday stress, coping).

In [ ]:
N_TOPICS = 5

# Fit vectorizer + LDA on training data only
count_vec = CountVectorizer(max_df=0.95, min_df=1, stop_words=None, max_features=500)
dtm_train = count_vec.fit_transform(X_train)
dtm_test  = count_vec.transform(X_test)

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=30,
    learning_method='batch',
    random_state=42
)
lda.fit(dtm_train)

X_lda_train = lda.transform(dtm_train)   # shape: (n_train, N_TOPICS)
X_lda_test  = lda.transform(dtm_test)

print(f'LDA train shape: {X_lda_train.shape}')
print(f'LDA test shape : {X_lda_test.shape}')

In [ ]:
# Inspect top words per LDA topic
vocab = count_vec.get_feature_names_out()
for idx, topic in enumerate(lda.components_):
    top_words = [vocab[i] for i in topic.argsort()[-10:][::-1]]
    print(f'Topic {idx}: {", ".join(top_words)}')

## 5. Ensemble — Late Fusion

Strategy: concatenate `[BERT_emb | LDA_probs]` and train a logistic regression meta-classifier.

```
text ──► BERT encoder ──► 768-dim embedding ──┐
                                               ├──► concat ──► LogReg ──► label
text ──► CountVec + LDA ──► 5-dim topics  ────┘
```

In [ ]:
# Concatenate features
X_ensemble_train = np.hstack([X_bert_train, X_lda_train])
X_ensemble_test  = np.hstack([X_bert_test,  X_lda_test])

print(f'Ensemble train shape: {X_ensemble_train.shape}')  # 768 + 5 = 773

# Scale (important for logistic regression on high-dim BERT features)
scaler = StandardScaler()
X_ensemble_train_sc = scaler.fit_transform(X_ensemble_train)
X_ensemble_test_sc  = scaler.transform(X_ensemble_test)

# Meta-classifier
meta_clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_clf.fit(X_ensemble_train_sc, y_train)

y_pred = meta_clf.predict(X_ensemble_test_sc)
y_prob = meta_clf.predict_proba(X_ensemble_test_sc)[:, 1]

print('\nClassification Report (Ensemble):')
print(classification_report(y_test, y_pred, target_names=['no_risk', 'suicide_risk']))

## 6. Comparison — BERT-only vs LDA-only vs Ensemble

In [ ]:
results = {}

for name, X_tr, X_te in [
    ('BERT only',   scaler.fit_transform(np.hstack([X_bert_train, np.zeros((len(X_bert_train), N_TOPICS))])),
                    scaler.transform(np.hstack([X_bert_test,  np.zeros((len(X_bert_test), N_TOPICS))]))),
    ('LDA only',    scaler.fit_transform(np.hstack([np.zeros((len(X_lda_train), 768)), X_lda_train])),
                    scaler.transform(np.hstack([np.zeros((len(X_lda_test), 768)), X_lda_test]))),
    ('Ensemble',    X_ensemble_train_sc, X_ensemble_test_sc),
]:
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    clf.fit(X_tr, y_train)
    prob = clf.predict_proba(X_te)[:, 1]
    pred = clf.predict(X_te)
    auc  = roc_auc_score(y_test, prob) if len(set(y_test)) > 1 else float('nan')
    from sklearn.metrics import f1_score, accuracy_score
    results[name] = {
        'accuracy': accuracy_score(y_test, pred),
        'f1':       f1_score(y_test, pred, zero_division=0),
        'auc':      auc,
        'prob':     prob,
    }

comparison = pd.DataFrame({k: {m: v for m, v in vals.items() if m != 'prob'}
                           for k, vals in results.items()}).T.round(3)
print(comparison)

## 7. Visualizations

In [ ]:
# --- 7a. ROC curves ---
fig, ax = plt.subplots(figsize=(6, 5))
colors = {'BERT only': 'steelblue', 'LDA only': 'tomato', 'Ensemble': 'seagreen'}

for name, vals in results.items():
    RocCurveDisplay.from_predictions(
        y_test, vals['prob'],
        name=f"{name} (AUC={vals['auc']:.2f})",
        ax=ax, color=colors[name]
    )

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_title('ROC Curves — BERT vs LDA vs Ensemble')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 7b. Confusion matrix (Ensemble) ---
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['no_risk', 'suicide_risk'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['no_risk', 'suicide_risk'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Ensemble)')

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 7c. Topic distribution by class (LDA topics) ---
df_train = pd.DataFrame(X_lda_train, columns=[f'Topic {i}' for i in range(N_TOPICS)])
df_train['label'] = y_train

topic_means = df_train.groupby('label').mean()

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(N_TOPICS)
w = 0.35
ax.bar(x - w/2, topic_means.loc[0], w, label='No Risk',       color='steelblue', alpha=0.8)
ax.bar(x + w/2, topic_means.loc[1], w, label='Suicide Risk',  color='tomato',    alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'Topic {i}' for i in range(N_TOPICS)])
ax.set_ylabel('Mean topic probability')
ax.set_title('LDA Topic Distribution by Class')
ax.legend()
plt.tight_layout()
plt.savefig('lda_topic_by_class.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 7d. Metrics comparison bar chart ---
fig, ax = plt.subplots(figsize=(7, 4))
metrics = ['accuracy', 'f1', 'auc']
x = np.arange(len(metrics))
w = 0.25
model_colors = ['steelblue', 'tomato', 'seagreen']

for i, (model, row) in enumerate(comparison.iterrows()):
    ax.bar(x + i*w, row[metrics], w, label=model, color=model_colors[i], alpha=0.85)

ax.set_xticks(x + w)
ax.set_xticklabels(['Accuracy', 'F1-score', 'AUC'])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Inference on New Texts

In [ ]:
def predict_risk(texts_raw: list[str]) -> pd.DataFrame:
    """Run full pipeline on new texts and return risk probabilities."""
    cleaned = [clean(t) for t in texts_raw]

    # BERT features
    bert_feat = get_bert_embeddings(cleaned)

    # LDA features
    dtm_new   = count_vec.transform(cleaned)
    lda_feat  = lda.transform(dtm_new)

    # Concatenate + scale
    combined  = np.hstack([bert_feat, lda_feat])
    combined_sc = scaler.transform(combined)

    probs  = meta_clf.predict_proba(combined_sc)[:, 1]
    preds  = meta_clf.predict(combined_sc)

    return pd.DataFrame({
        'text':         texts_raw,
        'prediction':   ['suicide_risk' if p == 1 else 'no_risk' for p in preds],
        'risk_prob':    probs.round(3)
    })


new_texts = [
    "Ya no puedo más, he pensado en hacerme daño esta noche.",
    "Tuve un mal día pero mañana espero que mejore.",
    "Me siento completamente vacío y sin ganas de seguir.",
    "Estoy nervioso por los exámenes pero confío en aprobar.",
]

result = predict_risk(new_texts)
pd.set_option('display.max_colwidth', 70)
result

## 9. Architecture Summary

```
                 ┌──────────────────────────────┐
                 │         Raw Text              │
                 └────────────┬─────────────────┘
                              │
              ┌───────────────┴───────────────┐
              ▼                               ▼
   ┌─────────────────────┐       ┌─────────────────────┐
   │  BERT (mBERT)       │       │  CountVectorizer     │
   │  Mean-pool          │       │  + LDA (5 topics)    │
   │  768-dim emb        │       │  5-dim topic probs   │
   └──────────┬──────────┘       └──────────┬──────────┘
              └────────────┬────────────────┘
                           ▼
              ┌─────────────────────────┐
              │  Concatenation (773-d)   │
              │  + StandardScaler        │
              └──────────┬──────────────┘
                         ▼
              ┌─────────────────────────┐
              │  Logistic Regression     │
              │  Meta-Classifier         │
              └──────────┬──────────────┘
                         ▼
              no_risk  /  suicide_risk
```

### Next steps for production
- Replace synthetic corpus with real annotated data (e.g., SuicideWatch Reddit dataset or your clinical records)
- Fine-tune BERT end-to-end instead of using frozen embeddings (much stronger signal)
- Use `dccuchile/bert-base-spanish-wwm-cased` (BETO) for Spanish-only text
- Add calibration (`CalibratedClassifierCV`) to improve probability estimates
- Evaluate with cross-validation on larger data to reduce variance